In [ ]:
%%sql -r dataframe_1
USE ROLE DOG_DATA5035_ROLE; 

# MOAB Rover Survey Lab: Feature Engineering in Snowflake

## Goal
In this lab, we will engineer new features from rover-collected survey data using:
- Python UDFs
- SQL views
- AI prompts

## Raw Inputs
- Easting
- Northing
- Sensor measurement

## Engineered Features
1. Grid tile, survey unit, and subcell assignment
2. Tile-level aggregated measurement signals
3. Comparison to normal range
4. Remediation prioritization

## SCALARS Mapping
- **Simplify**: convert coordinates into tile IDs
- **Aggregate**: summarize measurements at the tile level using multi-level aggregation
- **Assess**: compare readings to expected range
- **Rank / Score**: prioritize areas for remediation

In [ ]:
%%sql -r source_data
-- Change to your user's schema
USE SCHEMA data5035.DOG;
SELECT * FROM data5035.spring26.sdg_001_ra226_scandata limit 10;

## Convert from Coordinates to Grid

The input data is provided in directional distances on a flat map projection. Northing and Easting indicate how far to go in those directions (up and right) in US Feet relative to a known starting point. Our purpose here is to map those directional distances on to a three-level grid.

### Measurement
* **Tiles** are the largest areas. They are composed of a grid of **Survey Units** 21 tiles wide x 18 tiles tall.
* **Survey Units** measure 32.81 ft x 32.81 ft square
* **Subcells** are square subdivisions within the **Survey Units** laid out 10x10

### Labeling
* **Tiles** are coded by row letter and column letter starting at AA in the bottom-left of our map, given some known origin (2180160.001, 6660000.000). AA indicates 1st row, 1st column. AB indicates 1st row, 2nd column to the left. BA indicates 2nd row up, 1st column.
* Within each Tile, **Survey Units** are numbered starting in the top-left corner, proceeeding right, then down to the beginning of the next row (as if you're reading down a page)
* Within each Survey Unit, **Subcells** are numbered starting in the bottom-left corner, proceeduing right, then up to the beginning of thext row (as if you're reading from the bottom of a page up)

**Create a Python UDF to convert x (easting), y (northing) into the grid labels.**

### Testing

```
    >>> convert_xy(2180160.0001, 6660000.0000)  
    ('AA', 358, 1)

    >>> convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000)
    ('AB', 358, 1)

    >>> convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)
    ('BB', 338, 12)
```

In [ ]:
%%sql -r dataframe_2
CREATE OR REPLACE FUNCTION CONVERT_XY(
        X FLOAT,
        Y FLOAT,
        ORIGIN_X FLOAT,
        ORIGIN_Y FLOAT,
        SU_SIZE FLOAT,
        TILE_GRID_X NUMBER(38,0),
        TILE_GRID_Y NUMBER(38,0),
        SUBCELL_GRID NUMBER(38,0)
    )
    RETURNS OBJECT
    LANGUAGE PYTHON
    RUNTIME_VERSION = '3.11'
    HANDLER = 'convert_xy'
    AS 
    $$
import math

def convert_xy(x, y, origin_x, origin_y, su_size, tile_grid_x, tile_grid_y, subcell_grid):
    dx = x - origin_x
    dy = y - origin_y

    su_x = dx / su_size
    su_y = dy / su_size

    tile_col = int(math.floor(su_x / tile_grid_x))
    tile_row = int(math.floor(su_y / tile_grid_y))

    local_su_col = int(math.floor(su_x - tile_col * tile_grid_x))
    local_su_row = int(math.floor(su_y - tile_row * tile_grid_y))

    local_su_row_from_top = (tile_grid_y - 1) - local_su_row
    su_number = local_su_row_from_top * tile_grid_x + local_su_col + 1

    frac_x = (su_x - math.floor(su_x)) * subcell_grid
    frac_y = (su_y - math.floor(su_y)) * subcell_grid
    subcell_col = int(math.floor(frac_x))
    subcell_row = int(math.floor(frac_y))
    subcell = subcell_row * subcell_grid + subcell_col + 1

    row_letter = chr(ord('A') + tile_row)
    col_letter = chr(ord('A') + tile_col)
    tile_label = row_letter + col_letter

    return {"tile": tile_label, "su": su_number, "subcell": subcell}
    $$;

In [ ]:
%%sql -r dataframe_3
    CREATE OR REPLACE FUNCTION CONVERT_XY(X FLOAT, Y FLOAT)
    RETURNS OBJECT
    LANGUAGE SQL
    AS
    $$
        SELECT CONVERT_XY(
            X, Y,
            2180160.0001,
            6660000.0000,
            32.81,
            21,
            18,
            10
        )
    $$;

In [ ]:
%%sql -r dataframe_4
select convert_xy(2180160.0001, 6660000.0000);

In [ ]:
%%sql -r dataframe_5
select convert_xy(2180160.0001 + 32.81*21.01, 6660000.0000);

In [ ]:
%%sql -r dataframe_6
select convert_xy(2180160.0001 + 32.81*22.01 + 4, 6660000.0000 + 32.81*19.01 + 4)

In [ ]:
%%sql -r dataframe_7
SELECT 
    convert_xy(easting, northing) AS coordinates, 
    coordinates:su::INTEGER AS su,
    coordinates:subcell::INTEGER AS subcell,
    coordinates:tile::STRING AS tile,
    * 
FROM 
    data5035.spring26.sdg_001_ra226_scandata 
LIMIT 100;

## Compute Layered Averages

In [ ]:
%%sql -r dataframe_8
SELECT
convert_xy(easting,northing) AS coordinates, 
coordinates:tile::STRING AS tile,
coordinates:su::INTEGER AS su, 
coordinates:subcell::INTEGER AS subcell,
avg(reading)
FROM data5035.spring26.sdg_001_ra226_scandata
GROUP BY ALL
ORDER BY 2,3,4;

## Compare to Reference Ranges

This measurement is of Radium-226 levels.
* `<5` - OK
* `5 <= X < 7.4` - Warning
* `>= 7.4` - Alarm

In [ ]:
-- Assess each subcell's average Ra-226 reading against regulatory thresholds.
-- We first compute the per-subcell average in the inner CTE (subcell_avgs),
-- then classify each result and roll up a summary count per tile.

WITH subcell_avgs AS (
    -- Step 1: Average all raw readings down to the subcell grain
    SELECT
        convert_xy(easting, northing)    AS coordinates,
        coordinates:tile::STRING         AS tile,
        coordinates:su::INTEGER          AS su,
        coordinates:subcell::INTEGER     AS subcell,
        AVG(reading)                     AS avg_reading
    FROM data5035.spring26.sdg_001_ra226_scandata
    GROUP BY ALL
),

classified AS (
    -- Step 2: Apply the Ra-226 reference thresholds to label each subcell
    SELECT
        tile,
        su,
        subcell,
        avg_reading,
        CASE
            WHEN avg_reading < 5   THEN 'OK'       -- below action level
            WHEN avg_reading < 7.4 THEN 'Warning'  -- elevated; monitor closely
            ELSE                        'Alarm'    -- at or above remediation threshold
        END AS status
    FROM subcell_avgs
)

-- Step 3: Summarise status counts per tile so decision-makers can
-- quickly identify which tiles need the most attention.
SELECT
    tile,
    COUNT(*)                                       AS total_subcells,
    COUNT_IF(status = 'OK')                        AS ok_count,
    COUNT_IF(status = 'Warning')                   AS warning_count,
    COUNT_IF(status = 'Alarm')                     AS alarm_count,
    ROUND(AVG(avg_reading), 4)                     AS tile_avg_reading,
    ROUND(MAX(avg_reading), 4)                     AS tile_max_reading
FROM classified
GROUP BY tile
ORDER BY alarm_count DESC, warning_count DESC;

## Prioritize

Once we've build everything out, let's use AI to make some recommendations on remediation priorities.

In [ ]:
%%sql -r dataframe_10
-- Use Snowflake Cortex (COMPLETE) to generate plain-language remediation
-- recommendations for each tile. We build a structured prompt that includes
-- the tile's key statistics and status counts, then ask the LLM to rank
-- urgency and suggest next steps.

WITH subcell_avgs AS (
    -- Re-compute subcell averages (same logic as previous cell)
    SELECT
        convert_xy(easting, northing)    AS coordinates,
        coordinates:tile::STRING         AS tile,
        coordinates:su::INTEGER          AS su,
        coordinates:subcell::INTEGER     AS subcell,
        AVG(reading)                     AS avg_reading
    FROM data5035.spring26.sdg_001_ra226_scandata
    GROUP BY ALL
),

classified AS (
    -- Apply reference thresholds
    SELECT
        tile, su, subcell, avg_reading,
        CASE
            WHEN avg_reading < 5   THEN 'OK'
            WHEN avg_reading < 7.4 THEN 'Warning'
            ELSE                        'Alarm'
        END AS status
    FROM subcell_avgs
),

tile_summary AS (
    -- Roll up to tile-level stats used to build the AI prompt
    SELECT
        tile,
        COUNT(*)                     AS total_subcells,
        COUNT_IF(status = 'OK')      AS ok_count,
        COUNT_IF(status = 'Warning') AS warning_count,
        COUNT_IF(status = 'Alarm')   AS alarm_count,
        ROUND(AVG(avg_reading), 4)   AS tile_avg_reading,
        ROUND(MAX(avg_reading), 4)   AS tile_max_reading
    FROM classified
    GROUP BY tile
)

-- Pass each tile's summary stats into Cortex COMPLETE to generate a
-- concise, actionable remediation recommendation.
SELECT
    tile,
    alarm_count,
    warning_count,
    tile_avg_reading,
    tile_max_reading,
    SNOWFLAKE.CORTEX.COMPLETE(
        'mistral-large',
        CONCAT(
            'You are an environmental remediation specialist. ',
            'A rover survey measured Radium-226 levels (pCi/g) across grid tile ', tile, '. ',
            'Thresholds: OK < 5, Warning 5-7.4, Alarm >= 7.4. ',
            'Tile statistics: ',
              total_subcells, ' total subcells; ',
              ok_count,      ' OK; ',
              warning_count, ' Warning; ',
              alarm_count,   ' Alarm. ',
            'Average reading: ', tile_avg_reading, ' pCi/g. ',
            'Maximum reading: ', tile_max_reading, ' pCi/g. ',
            'In 2-3 sentences: assess the urgency for this tile and recommend the immediate next steps for remediation.'
        )
    ) AS ai_recommendation
FROM tile_summary
ORDER BY alarm_count DESC, warning_count DESC;